In [0]:
def readStationsBronze():
    from pyspark.sql import functions as F
    df_stations_silver = (spark.readStream
                            .format("delta")
                            .table("ev_spark.bronze.alt_fuel_stns")
                            .filter(F.col("Latitude").isNotNull() & F.col("Longitude").isNotNull())
                            .filter(F.col("Status_Code")== "E")
                            .withColumn("fsa", F.substring(F.col("ZIP"), 1, 3))
                            .select("Station_Name", "fsa", "Latitude", "Longitude", "EV_Level1_EVSE_Num", "EV_Level2_EVSE_Num", "EV_DC_Fast_Count")
    )
    df_stations_silver = (df_stations_silver
                            .withColumnRenamed("EV_Level1_EVSE_Num", "EV_L1_Count")
                            .withColumnRenamed("EV_Level2_EVSE_Num", "EV_L2_Count")
                            .withColumnRenamed("EV_DC_Fast_Count", "EV_L3_Count")
                            .withColumnRenamed("Station_Name", "Station")
                            .withColumn("EV_L1_Count", F.col("EV_L1_Count").cast("int"))
                            .withColumn("EV_L2_Count", F.col("EV_L2_Count").cast("int"))
                            .withColumn("EV_L3_Count", F.col("EV_L3_Count").cast("int"))
                            .fillna(0, subset= ['EV_L1_Count', 'EV_L2_Count', 'EV_L3_Count'])
                            )
    
    return df_stations_silver


In [0]:
def writeStationsSilver(df):
    (df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/ev_spark/myvol/checkpoint/chkpt/alt_fuel_stns_silver/")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable("ev_spark.silver.alt_fuel_stns")
    )

In [0]:
stations_df = readStationsBronze()
writeStationsSilver(stations_df)